In [ ]:
# eval_many.py
import os, sys, csv, argparse, subprocess, shlex, itertools
from pathlib import Path


In [ ]:

def boolstr(x): 
    return "true" if str(x).lower() in ("1","true","t","yes","y") else "false"

def load_manifest_rows(path):
    with open(path) as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise ValueError("manifest is empty")
    return rows

def cartesian_from_config(cfg_path, fixed_overrides):
    try:
        import yaml
    except ImportError:
        print("pip install pyyaml が必要です", file=sys.stderr); raise
    cfg = yaml.safe_load(open(cfg_path))
    params = cfg.get("parameters", {})
    # grid 値だけ拾う（values: [...] の形）
    grid = {k: v.get("values", v) for k, v in params.items()}
    # 評価に必要なキーだけに絞る & デフォルト
    keys = ["L","T","ch","N","gamma","J","use_omega","global_omg","learn_omg","K","data"]
    defaults = dict(L=1, T=32, ch=512, N=4, gamma=0.01, J="attn",
                    use_omega=True, global_omg=True, learn_omg=False,
                    K=1, data="ood")
    space = {k: grid.get(k, [defaults[k]]) for k in keys}
    # 上書き（例: --override K=16 data=ood）
    for k,v in fixed_overrides.items():
        if k in space:
            space[k] = [v]
    # 直積を辞書で
    for vals in itertools.product(*space.values()):
        yield dict(zip(space.keys(), vals))

def build_cmd(row, project, entity):
    ckpt = row["ckpt"]
    L = row.get("L","1"); T=row.get("T","32"); ch=row.get("ch","512"); N=row.get("N","4")
    gamma=row.get("gamma","0.01"); J=row.get("J","attn")
    use_omega=boolstr(row.get("use_omega","true"))
    global_omg=boolstr(row.get("global_omg","true"))
    learn_omg=boolstr(row.get("learn_omg","false"))
    data=row.get("data","ood")
    K=row.get("K","1")
    run_name = row.get("run_name", f"eval_L{L}_T{T}_ch{ch}_N{N}_K{K}_job{{job_id}}")
    cmd = [
        "python","eval_sudoku_wandb.py",
        "--model_path", ckpt,
        "--model","akorn",
        "--L", str(L), "--T", str(T), "--ch", str(ch), "--N", str(N),
        "--gamma", str(gamma), "--J", str(J),
        "--use_omega", use_omega, "--global_omg", global_omg, "--learn_omg", learn_omg,
        "--data", data, "--K", str(K),
        "--wandb_project", project, "--wandb_entity", entity,
        "--wandb_run_name", run_name
    ]
    return cmd


In [ ]:

def main():
    p = argparse.ArgumentParser()
    # 方式A
    p.add_argument("--manifest", type=str, help="評価行を並べたCSV (ckpt, L, T, ch, ...)")
    # 方式B
    p.add_argument("--config", type=str, help="sweep用のYAML (parameters.*.values を使用)")
    p.add_argument("--ckpt_template", type=str, help="ckptテンプレ (例: /ckpts/L{L}_T{T}_ch{ch}/ema_99.pth)")
    p.add_argument("--override", nargs="*", default=[], help="key=val を空間に上書き (例: K=16 data=ood)")
    # 分散用
    p.add_argument("--row_mod", type=int, default=1)
    p.add_argument("--row_rem", type=int, default=0)
    # その他
    p.add_argument("--dry_run", action="store_true")
    args = p.parse_args()

    # PJM→PBSジョブID互換（あなたのコードが job_id を参照するため）
    if "PJM_JOBID" in os.environ and "PBS_JOBID" not in os.environ:
        os.environ["PBS_JOBID"] = os.environ["PJM_JOBID"]

    project = os.environ.get("WANDB_PROJECT", "sudoku_akorn_eval")
    entity  = os.environ.get("WANDB_ENTITY", "shunsuke-kamiya-the-university-of-tokyo")

    rows = []
    if args.manifest:
        rows = load_manifest_rows(args.manifest)
    else:
        if not (args.config and args.ckpt_template):
            print("Either --manifest OR (--config AND --ckpt_template) を指定してください。", file=sys.stderr)
            sys.exit(2)
        overrides = {}
        for kv in args.override:
            k,v = kv.split("=",1)
            overrides[k] = v
        for combo in cartesian_from_config(args.config, overrides):
            # テンプレから ckpt パスを生成
            ckpt = args.ckpt_template.format(**{k:str(v) for k,v in combo.items()})
            row = dict(combo)
            row["ckpt"] = ckpt
            rows.append(row)

    total = len(rows)
    processed = 0
    for idx, r in enumerate(rows):
        if idx % args.row_mod != args.row_rem:
            continue
        # 存在チェック（テンプレ方式では重要）
        if not Path(r["ckpt"]).exists():
            print(f"[skip] ckpt not found: {r['ckpt']}", file=sys.stderr)
            continue
        cmd = build_cmd(r, project, entity)
        print(f"[{idx+1}/{total}] " + " ".join(shlex.quote(c) for c in cmd), flush=True)
        if not args.dry_run:
            ret = subprocess.run(cmd)
            if ret.returncode != 0:
                print(f"!! row {idx} failed (ckpt={r['ckpt']})", file=sys.stderr)
        processed += 1
    print(f"done. processed={processed}/{total}, shard=({args.row_rem} of {args.row_mod})")

if __name__ == "__main__":
    main()
